<a href="https://colab.research.google.com/github/cybercolombia/suelosabio/blob/feature%2FSCRUM-16/notebooks/ClimatePipeline/04_Climate_TemperaturaMinima_DailyAudit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Climate_TemperaturaMinima_DailyAudit

Audita la capa preliminar `clima_diario_sensor` sin modificarla.

## Objetivos

- Construir un calendario explícito por estación y sensor sin convertir ausencias en cero.
- Evaluar un umbral de cobertura como candidato, no como regla aprobada.
- Detectar coberturas superiores a la tolerancia configurada y valores que requieren revisión.
- Comparar sensores paralelos sin sumarlos ni escoger todavía uno como canónico.
- Exportar evidencia reproducible para decidir las reglas del notebook 05.

Este notebook **no corrige, imputa, elimina ni acepta** observaciones. Los extremos son candidatos para revisión, no errores automáticos.

## 1. Preparar el repositorio en Colab

La celda clona o actualiza la rama y carga los módulos versionados. Antes de la entrega final se debe reemplazar la rama por un commit fijo.

In [ ]:
from pathlib import Path

import subprocess
import sys

try:
    import google.colab  # noqa: F401

    IN_COLAB = True
except ImportError:
    IN_COLAB = False

REPO_URL = 'https://github.com/cybercolombia/suelosabio.git'
REPO_REF = 'feature/SCRUM-16'
REPO_DIR = Path('/content/suelosabio') if IN_COLAB else Path.cwd()
ACTUALIZAR_REPOSITORIO = True

if IN_COLAB:
    if not (REPO_DIR / '.git').exists():
        subprocess.run(
            ['git', 'clone', '--depth', '1', '--branch', REPO_REF, REPO_URL, str(REPO_DIR)],
            check=True,
        )
    elif ACTUALIZAR_REPOSITORIO:
        subprocess.run(['git', 'fetch', 'origin', REPO_REF], cwd=REPO_DIR, check=True)
        subprocess.run(['git', 'checkout', REPO_REF], cwd=REPO_DIR, check=True)
        subprocess.run(['git', 'pull', '--ff-only', 'origin', REPO_REF], cwd=REPO_DIR, check=True)

PIPELINE_DIR = REPO_DIR / 'notebooks' / 'ClimatePipeline'
if not PIPELINE_DIR.exists():
    raise FileNotFoundError(f'No existe la carpeta del pipeline: {PIPELINE_DIR}')
if str(PIPELINE_DIR) not in sys.path:
    sys.path.insert(0, str(PIPELINE_DIR))

print({'in_colab': IN_COLAB, 'repo_ref': REPO_REF, 'repo_dir': str(REPO_DIR)})

## 2. Configuración protegida

La configuración inicial reproduce los cuatro pilotos de enero y febrero de 2025. Los umbrales son hipótesis de auditoría: marcarlos no equivale a eliminar datos ni aprobar días.

In [ ]:
import json
import time

import pandas as pd

from ClimateProcessingUtils import (
    ahora_proyecto,
    construir_plan_particiones,
    detectar_commit,
    escribir_json_atomico,
    escribir_parquet_atomico,
    escribir_texto_atomico,
    formatear_duracion,
    ruta_particion_diaria,
    slugificar,
)
from DatasetConfig import cargar_configuracion_datasets
try:
    from IPython.display import Markdown, display
except ImportError:
    Markdown = str

    def display(valor):
        print(valor)

VARIABLE_NOMBRE = 'temperatura_minima'
DATASET_ID = 'afdg-3zpb'
AUDITAR_DEPARTAMENTOS = ['CUNDINAMARCA', 'BOYACÁ']
AUDITAR_ANIOS = [2024, 2025]
AUDITAR_MESES = list(range(1, 13))
AUDITORIA_NOMBRE = 'cierre_temperatura_minima_2024_2025_v1'

UMBRAL_COBERTURA_CANDIDATO_PCT = 90.0
TOLERANCIA_COBERTURA_SUPERIOR_PCT = 102.0
UMBRAL_TOTAL_EXTREMO_MM = 200.0
UMBRAL_INTERVALO_SOSPECHOSO_MM = 25.0
TOLERANCIA_SENSORES_MM = 0.1
UMBRAL_TEMPERATURA_MIN_C = -10.0
UMBRAL_TEMPERATURA_MAX_C = 45.0
UMBRAL_AMPLITUD_TERMICA_C = 25.0
TOLERANCIA_SENSORES_C = 1.0

GUARDAR_RESULTADOS = True
SOBRESCRIBIR_AUDITORIA = False
EJECUTAR_AUDITORIA_DIARIA = False

DATASET_CONFIG = cargar_configuracion_datasets(in_colab=IN_COLAB)
PROCESSED_ROOT = DATASET_CONFIG.processed_root

if VARIABLE_NOMBRE == 'precipitacion':
    from PrecipitationDailyAudit import AUDIT_VERSION, auditar_precipitacion_diaria

    def AUDITAR_VARIABLE(diario):
        return auditar_precipitacion_diaria(
            diario,
            umbral_cobertura_pct=UMBRAL_COBERTURA_CANDIDATO_PCT,
            umbral_total_extremo_mm=UMBRAL_TOTAL_EXTREMO_MM,
            umbral_intervalo_sospechoso_mm=UMBRAL_INTERVALO_SOSPECHOSO_MM,
            tolerancia_sensores_mm=TOLERANCIA_SENSORES_MM,
        )

    PARAMETROS_AUDITORIA = {
        'umbral_cobertura_candidato_pct': UMBRAL_COBERTURA_CANDIDATO_PCT,
        'umbral_total_extremo_mm': UMBRAL_TOTAL_EXTREMO_MM,
        'umbral_intervalo_sospechoso_mm': UMBRAL_INTERVALO_SOSPECHOSO_MM,
        'tolerancia_sensores_mm': TOLERANCIA_SENSORES_MM,
    }
    TITULO_REPORTE = 'Auditoría diaria de precipitación'
    COLUMNA_VALOR_GRAFICA = 'precipitacion_observada_mm'
    ETIQUETA_VALOR_GRAFICA = 'Precipitación observada (mm)'
elif VARIABLE_NOMBRE in {
    'temperatura_ambiente',
    'temperatura_minima',
    'temperatura_maxima',
}:
    from TemperatureDailyAudit import AUDIT_VERSION, auditar_temperatura_diaria

    def AUDITAR_VARIABLE(diario):
        return auditar_temperatura_diaria(
            diario,
            umbral_cobertura_pct=UMBRAL_COBERTURA_CANDIDATO_PCT,
            tolerancia_cobertura_superior_pct=TOLERANCIA_COBERTURA_SUPERIOR_PCT,
            umbral_minimo_c=UMBRAL_TEMPERATURA_MIN_C,
            umbral_maximo_c=UMBRAL_TEMPERATURA_MAX_C,
            umbral_amplitud_c=UMBRAL_AMPLITUD_TERMICA_C,
            tolerancia_sensores_c=TOLERANCIA_SENSORES_C,
        )

    PARAMETROS_AUDITORIA = {
        'umbral_cobertura_candidato_pct': UMBRAL_COBERTURA_CANDIDATO_PCT,
        'tolerancia_cobertura_superior_pct': TOLERANCIA_COBERTURA_SUPERIOR_PCT,
        'umbral_temperatura_min_c': UMBRAL_TEMPERATURA_MIN_C,
        'umbral_temperatura_max_c': UMBRAL_TEMPERATURA_MAX_C,
        'umbral_amplitud_termica_c': UMBRAL_AMPLITUD_TERMICA_C,
        'tolerancia_sensores_c': TOLERANCIA_SENSORES_C,
    }
    TITULO_REPORTE = f'Auditoría diaria de {VARIABLE_NOMBRE}'
    COLUMNA_VALOR_GRAFICA = 'temperatura_principal_observada_c'
    ETIQUETA_VALOR_GRAFICA = 'Temperatura principal observada (°C)'
elif VARIABLE_NOMBRE == 'humedad':
    from HumidityDailyAudit import detener_auditoria_pendiente

    detener_auditoria_pendiente()
elif VARIABLE_NOMBRE == 'presion_atmosferica':
    from AtmosphericPressureDailyAudit import detener_auditoria_pendiente

    detener_auditoria_pendiente()
elif VARIABLE_NOMBRE == 'velocidad_viento':
    from WindSpeedDailyAudit import detener_auditoria_pendiente

    detener_auditoria_pendiente()
else:
    raise ValueError('La variable no tiene auditoría diaria implementada.')
AUDIT_OUTPUT_DIR = (
    PROCESSED_ROOT
    / 'auditorias_clima_diario'
    / f'variable={VARIABLE_NOMBRE}'
    / f'fuente={DATASET_ID}'
    / f'auditoria={slugificar(AUDITORIA_NOMBRE)}'
)

plan_particiones = construir_plan_particiones(
    VARIABLE_NOMBRE,
    DATASET_ID,
    AUDITAR_DEPARTAMENTOS,
    AUDITAR_ANIOS,
    AUDITAR_MESES,
)

print({
    'audit_version': AUDIT_VERSION,
    'particiones': len(plan_particiones),
    'ejecutar': EJECUTAR_AUDITORIA_DIARIA,
    'guardar': GUARDAR_RESULTADOS,
    'sobrescribir': SOBRESCRIBIR_AUDITORIA,
    'salida': str(AUDIT_OUTPUT_DIR),
})

In [ ]:
def estado_entrada(spec):
    input_dir = ruta_particion_diaria(PROCESSED_ROOT, spec)
    manifest_path = input_dir / 'manifest.json'
    daily_path = input_dir / 'observaciones_diarias.parquet'
    estado = 'NO_ENCONTRADA'
    filas = None
    commit = None
    worker = None
    if manifest_path.exists():
        manifest = json.loads(manifest_path.read_text(encoding='utf-8'))
        estado = manifest.get('estado', 'SIN_ESTADO')
        filas = manifest.get('metricas', {}).get('filas_diarias_salida')
        commit = manifest.get('commit')
        worker = manifest.get('worker_id')
        if estado == 'COMPLETA' and not daily_path.exists():
            estado = 'MANIFEST_SIN_PARQUET'
    return {
        **spec.como_dict(),
        'estado': estado,
        'filas_diarias': filas,
        'worker_id': worker,
        'commit': commit,
        'entrada': str(input_dir),
    }


plan_df = pd.DataFrame([estado_entrada(spec) for spec in plan_particiones])
display(Markdown(f'### Plan de auditoría: {len(plan_df)} particiones'))
display(plan_df)

## 3. Carga, análisis y exportación

Solo se aceptan particiones con manifiesto `COMPLETA`. La auditoría genera calendarios y reportes en una carpeta hermana; nunca escribe dentro de `clima_diario_sensor`.

In [ ]:
NOMBRES_SALIDA = {
    'calendario': 'calendario_estacion_sensor.parquet',
    'particiones': 'resumen_particiones.parquet',
    'pares': 'resumen_estacion_sensor.parquet',
    'catalogo': 'catalogo_estacion_sensor.parquet',
    'actividad_mensual': 'actividad_mensual_estacion_sensor.parquet',
    'ausencias_mes': 'ausencias_mes_completo.parquet',
    'sospechosos': 'valores_sospechosos.parquet',
    'comparaciones': 'comparaciones_sensores.parquet',
    'paralelos': 'resumen_sensores_paralelos.parquet',
    'reporte': (
        f'AuditoriaDiaria_{slugificar(VARIABLE_NOMBRE)}_'
        f'{slugificar(AUDITORIA_NOMBRE)}.md'
    ),
    'manifest': 'manifest.json',
}


def cargar_particiones_completas():
    bloques = []
    procedencia = []
    for spec in plan_particiones:
        input_dir = ruta_particion_diaria(PROCESSED_ROOT, spec)
        manifest_path = input_dir / 'manifest.json'
        daily_path = input_dir / 'observaciones_diarias.parquet'
        if not manifest_path.exists():
            raise FileNotFoundError(f'No existe el manifiesto: {manifest_path}')
        manifest = json.loads(manifest_path.read_text(encoding='utf-8'))
        if manifest.get('estado') != 'COMPLETA':
            raise RuntimeError(f'La partición no está completa: {input_dir}')
        if not daily_path.exists():
            raise FileNotFoundError(f'No existe la salida diaria: {daily_path}')
        tabla = pd.read_parquet(daily_path)
        filas_manifest = manifest.get('metricas', {}).get('filas_diarias_salida')
        if filas_manifest is not None and len(tabla) != int(filas_manifest):
            raise RuntimeError(
                f'Filas de {daily_path} ({len(tabla):,}) != manifiesto ({filas_manifest:,}).'
            )
        bloques.append(tabla)
        procedencia.append({
            **spec.como_dict(),
            'ruta': str(daily_path),
            'filas': len(tabla),
            'commit': manifest.get('commit'),
            'worker_id': manifest.get('worker_id'),
        })
    return pd.concat(bloques, ignore_index=True), procedencia


def tabla_markdown(tabla, columnas=None, limite=None):
    vista = tabla.loc[:, columnas] if columnas else tabla
    if limite is not None:
        vista = vista.head(limite)
    try:
        return vista.to_markdown(index=False)
    except ImportError:
        return '```text\n' + vista.to_string(index=False) + '\n```'


def construir_reporte(resultado, procedencia, inicio, fin, duracion):
    motivos = (
        resultado.valores_sospechosos['motivos_revision']
        .str.get_dummies(sep='|')
        .sum()
        .rename_axis('motivo')
        .reset_index(name='filas')
    ) if not resultado.valores_sospechosos.empty else pd.DataFrame(columns=['motivo', 'filas'])

    if VARIABLE_NOMBRE == 'precipitacion':
        top_columnas = [
            'departamento', 'codigoestacion', 'codigosensor', 'fecha',
            'precipitacion_observada_mm', 'valor_intervalo_max_mm',
            'cobertura_observada_pct', 'motivos_revision',
        ]
    else:
        top_columnas = [
            'departamento', 'codigoestacion', 'codigosensor', 'fecha',
            'temperatura_principal_observada_c',
            'temperatura_minima_observada_c',
            'temperatura_maxima_observada_c',
            'amplitud_termica_observada_c',
            'cobertura_observada_pct', 'motivos_revision',
        ]
    secciones_catalogo = []
    if hasattr(resultado, 'catalogo_pares'):
        secciones_catalogo = [
            '',
            '## Catálogo esperado de estaciones y sensores',
            '',
            f'- Pares en el catálogo: {len(resultado.catalogo_pares):,}.',
            f'- Ausencias de mes completo entre primera y última observación: {len(resultado.ausencias_mes_completo):,}.',
            '',
            '### Ausencias de mes completo',
            '',
            tabla_markdown(resultado.ausencias_mes_completo, limite=100),
        ]

    secciones = [
        f'# {TITULO_REPORTE}',
        '',
        f'- Versión: `{AUDIT_VERSION}`',
        f'- Inicio: `{inicio.isoformat()}`',
        f'- Fin: `{fin.isoformat()}`',
        f'- Duración: `{formatear_duracion(duracion)}`',
        f'- Commit auditor: `{detectar_commit(REPO_DIR)}`',
        f'- Umbral de cobertura candidato: `{UMBRAL_COBERTURA_CANDIDATO_PCT} %`',
        f'- Tolerancia superior de cobertura: `{TOLERANCIA_COBERTURA_SUPERIOR_PCT} %`',
        '',
        '> Los umbrales de este reporte son diagnósticos. Ninguna fila fue eliminada, imputada o aceptada automáticamente.',
        '',
        '## Procedencia',
        '',
        tabla_markdown(pd.DataFrame(procedencia)),
        '',
        '## Resumen por partición',
        '',
        tabla_markdown(resultado.resumen_particiones),
        *secciones_catalogo,
        '',
        '## Motivos de revisión',
        '',
        tabla_markdown(motivos),
        '',
        '## Valores candidatos para revisión',
        '',
        tabla_markdown(resultado.valores_sospechosos, top_columnas, limite=50),
        '',
        '## Sensores paralelos',
        '',
        tabla_markdown(resultado.resumen_sensores_paralelos),
        '',
        '## Lectura preliminar',
        '',
        '- Los días sin observación permanecen como `NaN`, no como cero.',
        '- Una cobertura mayor a la tolerancia superior exige revisar frecuencia y límites temporales.',
        '- Un extremo se marca para revisión; este notebook no declara que sea inválido.',
        '- Los sensores paralelos siguen separados hasta aprobar una regla de selección.',
        '',
    ]
    return '\n'.join(secciones)


def guardar_figuras(resultado, output_dir):
    try:
        import matplotlib.pyplot as plt
    except ImportError:
        print('Matplotlib no está disponible; se omiten las figuras.')
        return []

    figuras_dir = output_dir / 'figures'
    figuras_dir.mkdir(parents=True, exist_ok=True)
    rutas = []

    resumen = resultado.resumen_particiones.copy()
    resumen['particion'] = (
        resumen['departamento'].astype(str)
        + ' '
        + resumen['anio'].astype(str)
        + '-'
        + resumen['mes'].astype(str).str.zfill(2)
    )
    fig, ax = plt.subplots(figsize=(10, 5))
    ax.bar(resumen['particion'], resumen['dias_sin_ningun_registro'], color='#c6533d')
    ax.set_title('Días sin ningún registro en la partición')
    ax.set_ylabel('Días ausentes')
    ax.tick_params(axis='x', rotation=25)
    fig.tight_layout()
    ruta = figuras_dir / 'dias_ausentes_por_particion.png'
    fig.savefig(ruta, dpi=150)
    if IN_COLAB:
        plt.show()
    plt.close(fig)
    rutas.append(str(ruta))

    if not resultado.valores_sospechosos.empty:
        fig, ax = plt.subplots(figsize=(10, 5))
        for departamento, grupo in resultado.valores_sospechosos.groupby('departamento'):
            ax.scatter(
                grupo['fecha'],
                grupo[COLUMNA_VALOR_GRAFICA],
                label=departamento,
                alpha=0.75,
            )
        if VARIABLE_NOMBRE == 'precipitacion':
            ax.axhline(UMBRAL_TOTAL_EXTREMO_MM, color='#c6533d', linestyle='--')
        else:
            ax.axhline(UMBRAL_TEMPERATURA_MIN_C, color='#3973a8', linestyle='--')
            ax.axhline(UMBRAL_TEMPERATURA_MAX_C, color='#c6533d', linestyle='--')
        ax.set_title('Valores diarios candidatos para revisión')
        ax.set_ylabel(ETIQUETA_VALOR_GRAFICA)
        ax.legend()
        fig.tight_layout()
        ruta = figuras_dir / 'valores_sospechosos.png'
        fig.savefig(ruta, dpi=150)
        if IN_COLAB:
            plt.show()
        plt.close(fig)
        rutas.append(str(ruta))
    return rutas


def guardar_auditoria(resultado, procedencia, reporte, inicio, fin, duracion):
    manifest_path = AUDIT_OUTPUT_DIR / NOMBRES_SALIDA['manifest']
    if manifest_path.exists() and not SOBRESCRIBIR_AUDITORIA:
        existente = json.loads(manifest_path.read_text(encoding='utf-8'))
        if existente.get('estado') == 'COMPLETA':
            print(f'La auditoría ya está completa; no se sobrescribe: {AUDIT_OUTPUT_DIR}')
            return existente
    if AUDIT_OUTPUT_DIR.exists() and any(AUDIT_OUTPUT_DIR.iterdir()) and not SOBRESCRIBIR_AUDITORIA:
        raise RuntimeError(f'Existe una auditoría incompleta: {AUDIT_OUTPUT_DIR}')

    escribir_json_atomico(
        {
            'estado': 'INICIADA',
            'audit_version': AUDIT_VERSION,
            'inicio': inicio.isoformat(),
            'commit': detectar_commit(REPO_DIR),
        },
        manifest_path,
        sobrescribir=SOBRESCRIBIR_AUDITORIA,
    )
    tablas = {
        'calendario': resultado.calendario,
        'particiones': resultado.resumen_particiones,
        'pares': resultado.resumen_pares,
        'sospechosos': resultado.valores_sospechosos,
        'comparaciones': resultado.comparaciones_sensores,
        'paralelos': resultado.resumen_sensores_paralelos,
    }
    if hasattr(resultado, 'catalogo_pares'):
        tablas.update({
            'catalogo': resultado.catalogo_pares,
            'actividad_mensual': resultado.actividad_mensual,
            'ausencias_mes': resultado.ausencias_mes_completo,
        })
    salidas = {}
    for nombre, tabla in tablas.items():
        ruta = AUDIT_OUTPUT_DIR / NOMBRES_SALIDA[nombre]
        escribir_parquet_atomico(tabla, ruta, sobrescribir=SOBRESCRIBIR_AUDITORIA)
        salidas[nombre] = {'ruta': str(ruta), 'filas': len(tabla), 'bytes': ruta.stat().st_size}

    reporte_path = AUDIT_OUTPUT_DIR / NOMBRES_SALIDA['reporte']
    escribir_texto_atomico(reporte, reporte_path, sobrescribir=SOBRESCRIBIR_AUDITORIA)
    figuras = guardar_figuras(resultado, AUDIT_OUTPUT_DIR)
    manifest = {
        'estado': 'COMPLETA',
        'audit_version': AUDIT_VERSION,
        'commit': detectar_commit(REPO_DIR),
        'inicio': inicio.isoformat(),
        'fin': fin.isoformat(),
        'duracion_segundos': round(duracion, 2),
        'parametros': PARAMETROS_AUDITORIA,
        'procedencia': procedencia,
        'metricas': resultado.metricas,
        'salidas': salidas,
        'reporte': str(reporte_path),
        'figuras': figuras,
    }
    escribir_json_atomico(manifest, manifest_path, sobrescribir=True)
    print(f'Auditoría guardada en: {AUDIT_OUTPUT_DIR}')
    return manifest

## 4. Ejecución protegida

Revise primero el plan con `EJECUTAR_AUDITORIA_DIARIA=False`. Después cambie la bandera a `True`, ejecute nuevamente la celda de configuración para actualizar el valor en memoria y finalmente ejecute esta celda protegida. La exportación puede desactivarse independientemente con `GUARDAR_RESULTADOS=False`.

In [ ]:
resultado_auditoria = None

if not EJECUTAR_AUDITORIA_DIARIA:
    print('Auditoría desactivada. Revise el plan y active EJECUTAR_AUDITORIA_DIARIA.')
else:
    inicio = ahora_proyecto()
    reloj = time.perf_counter()
    diario, procedencia = cargar_particiones_completas()
    print(f'Filas diarias cargadas: {len(diario):,}')
    resultado_auditoria = AUDITAR_VARIABLE(diario)
    fin = ahora_proyecto()
    duracion = time.perf_counter() - reloj
    reporte = construir_reporte(resultado_auditoria, procedencia, inicio, fin, duracion)

    display(Markdown('## Resumen por partición'))
    display(resultado_auditoria.resumen_particiones)
    display(Markdown('## Sensores paralelos'))
    display(resultado_auditoria.resumen_sensores_paralelos)
    if hasattr(resultado_auditoria, 'ausencias_mes_completo'):
        display(Markdown('## Ausencias de mes completo'))
        display(resultado_auditoria.ausencias_mes_completo)
    display(Markdown('## Valores candidatos para revisión'))
    display(resultado_auditoria.valores_sospechosos.head(50))
    print(f'Duración: {formatear_duracion(duracion)}')

    if GUARDAR_RESULTADOS:
        guardar_auditoria(
            resultado_auditoria,
            procedencia,
            reporte,
            inicio,
            fin,
            duracion,
        )
    else:
        print('Resultados no guardados porque GUARDAR_RESULTADOS=False.')

## 5. Serie diaria interactiva opcional

Esta celda usa Plotly para recorrer dos años con zoom y selector temporal. Grafica un solo par estación-sensor para no mezclar instrumentos. La franja inferior diferencia días observados y ausentes; una ausencia nunca se representa como precipitación cero.

La bandera permanece en `False` para que **Run all** no genere figuras pesadas accidentalmente.

In [ ]:
EJECUTAR_GRAFICA_INTERACTIVA = False
GRAFICA_USAR_RESULTADO_EN_MEMORIA = False
GRAFICA_DEPARTAMENTO = 'CUNDINAMARCA'
GRAFICA_ESTACION = '3505500121'
GRAFICA_SENSOR = '0240'

if EJECUTAR_GRAFICA_INTERACTIVA:
    try:
        import plotly.graph_objects as go
        from plotly.subplots import make_subplots
    except ImportError as exc:
        raise ImportError(
            'Plotly no está disponible. En Colab ejecute: %pip install plotly'
        ) from exc

    if (
        GRAFICA_USAR_RESULTADO_EN_MEMORIA
        and 'resultado_auditoria' in globals()
        and resultado_auditoria is not None
    ):
        calendario_grafica = resultado_auditoria.calendario.copy()
    else:
        ruta_calendario = AUDIT_OUTPUT_DIR / 'calendario_estacion_sensor.parquet'
        if not ruta_calendario.exists():
            raise FileNotFoundError(
                'No hay auditoría cargada ni calendario guardado en '
                f'{ruta_calendario}.'
            )
        calendario_grafica = pd.read_parquet(ruta_calendario)

    serie_grafica = calendario_grafica.loc[
        calendario_grafica['departamento'].eq(GRAFICA_DEPARTAMENTO)
        & calendario_grafica['codigoestacion'].astype('string').eq(
            str(GRAFICA_ESTACION)
        )
        & calendario_grafica['codigosensor'].astype('string').eq(
            str(GRAFICA_SENSOR)
        )
    ].sort_values('fecha')
    if serie_grafica.empty:
        disponibles = calendario_grafica[
            ['departamento', 'codigoestacion', 'codigosensor']
        ].drop_duplicates().sort_values(
            ['departamento', 'codigoestacion', 'codigosensor']
        )
        display(disponibles.head(100))
        raise ValueError('El par estación-sensor configurado no existe.')

    observadas = serie_grafica.loc[serie_grafica['es_dia_observado']].copy()
    disponibilidad = serie_grafica['es_dia_observado'].astype(bool)
    colores_disponibilidad = disponibilidad.map(
        {True: '#2f6b3c', False: '#c4473a'}
    )

    figura = make_subplots(
        rows=2,
        cols=1,
        shared_xaxes=True,
        vertical_spacing=0.08,
        row_heights=[0.82, 0.18],
        subplot_titles=(ETIQUETA_VALOR_GRAFICA, 'Disponibilidad diaria'),
    )
    figura.add_trace(
        go.Scattergl(
            x=observadas['fecha'],
            y=observadas[COLUMNA_VALOR_GRAFICA],
            mode='lines+markers',
            name='Observado',
            line={'color': '#277da1', 'width': 1},
            marker={'color': '#174f6b', 'size': 4},
            customdata=observadas[
                ['cobertura_observada_pct', 'estado_cobertura_candidato']
            ].to_numpy(),
            hovertemplate=(
                'Fecha=%{x|%Y-%m-%d}<br>'
                'Valor=%{y:.2f}<br>'
                'Cobertura=%{customdata[0]:.2f}%<br>'
                'Estado=%{customdata[1]}<extra></extra>'
            ),
            connectgaps=False,
        ),
        row=1,
        col=1,
    )
    figura.add_trace(
        go.Scattergl(
            x=serie_grafica['fecha'],
            y=disponibilidad.astype(int),
            mode='markers',
            name='Disponibilidad',
            marker={'color': colores_disponibilidad, 'size': 5},
            text=disponibilidad.map(
                {True: 'Observado', False: 'Ausente (NaN)'}
            ),
            hovertemplate='Fecha=%{x|%Y-%m-%d}<br>%{text}<extra></extra>',
        ),
        row=2,
        col=1,
    )
    figura.update_yaxes(title_text=ETIQUETA_VALOR_GRAFICA, row=1, col=1)
    figura.update_yaxes(
        tickvals=[0, 1],
        ticktext=['Ausente', 'Observado'],
        range=[-0.4, 1.4],
        row=2,
        col=1,
    )
    figura.update_xaxes(
        rangeselector={
            'buttons': [
                {'count': 1, 'label': '1m', 'step': 'month', 'stepmode': 'backward'},
                {'count': 3, 'label': '3m', 'step': 'month', 'stepmode': 'backward'},
                {'count': 6, 'label': '6m', 'step': 'month', 'stepmode': 'backward'},
                {'count': 1, 'label': '1a', 'step': 'year', 'stepmode': 'backward'},
                {'step': 'all', 'label': 'Todo'},
            ]
        },
        row=1,
        col=1,
    )
    figura.update_xaxes(rangeslider={'visible': True}, row=2, col=1)
    figura.update_layout(
        title=(
            f'{VARIABLE_NOMBRE}: {GRAFICA_DEPARTAMENTO} · '
            f'{GRAFICA_ESTACION}/{GRAFICA_SENSOR}'
        ),
        height=680,
        hovermode='x unified',
        template='plotly_white',
        legend={'orientation': 'h', 'y': 1.08},
    )
    figura.show()
else:
    print('Gráfica interactiva desactivada. Active EJECUTAR_GRAFICA_INTERACTIVA.')

## 6. Compuerta hacia el notebook 05

Después de revisar los Parquet, el Markdown y las figuras se decidirán por separado:

1. El umbral mínimo de cobertura y su tolerancia superior.
2. La regla para días completamente ausentes.
3. El tratamiento de patrones compatibles con saturación o código de error.
4. La comparación y eventual selección de sensores paralelos.

Hasta entonces el valor diario aceptado (`precipitacion_diaria_mm` o `temperatura_diaria_c`) debe permanecer en `NaN`. El paso 05 específico de esta variable consolida los sensores aprobados a una fila estación-día.